In [1]:
import pandas as pd
import os
import json
from datetime import datetime
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [10]:
from google.colab import files
import pandas as pd

# завантаження локального файлу
uploaded = files.upload()
filename = list(uploaded.keys())[0]
print(f"Uploaded file: {filename}")

# читаємо CSV
df = pd.read_csv(filename)

print("Shape:", df.shape)
print(df["split"].value_counts())

Saving processed_v2.csv to processed_v2 (1).csv
Uploaded file: processed_v2 (1).csv
Shape: (109166, 8)
split
train    87369
test     10955
dev      10842
Name: count, dtype: int64


In [3]:
print("Колонки:")
print(df.columns)

print("\nРозподіл split:")
print(df["split"].value_counts())

Колонки:
Index(['split', 'sent_id', 'token_id', 'token', 'lemma', 'upos', 'head',
       'deprel'],
      dtype='object')

Розподіл split:
split
train    87369
test     10955
dev      10842
Name: count, dtype: int64


In [11]:
os.makedirs("data/splits", exist_ok=True)
os.makedirs("data/sample", exist_ok=True)
os.makedirs("docs", exist_ok=True)
os.makedirs("src", exist_ok=True)

In [12]:
train_df = df[df["split"]=="train"]
val_df   = df[df["split"]=="dev"]
test_df  = df[df["split"]=="test"]

train_df.to_csv("data/splits/train.csv",index=False)
val_df.to_csv("data/splits/validation.csv",index=False)
test_df.to_csv("data/splits/test.csv",index=False)

print("CSV split файли створено")

CSV split файли створено


In [6]:
train_ids = df[df["split"]=="train"]["sent_id"].unique()
val_ids   = df[df["split"]=="dev"]["sent_id"].unique()
test_ids  = df[df["split"]=="test"]["sent_id"].unique()

with open("data/sample/splits_train_ids.txt","w") as f:
    for i in train_ids:
        f.write(str(i)+"\n")

with open("data/sample/splits_val_ids.txt","w") as f:
    for i in val_ids:
        f.write(str(i)+"\n")

with open("data/sample/splits_test_ids.txt","w") as f:
    for i in test_ids:
        f.write(str(i)+"\n")

print("Split ID файли створено")

Split ID файли створено


In [14]:
sentences = (
    df.groupby(["sent_id","split"])["token"]
      .apply(lambda x: " ".join(x))
      .reset_index(name="text")
)

train_sent = sentences[sentences.split=="train"]
val_sent   = sentences[sentences.split=="dev"]
test_sent  = sentences[sentences.split=="test"]

print("sentences:",len(sentences))

sentences: 7142


In [15]:
train_text = set(train_sent.text)
val_text   = set(val_sent.text)
test_text  = set(test_sent.text)

dup_train_test = len(train_text & test_text)
dup_train_val  = len(train_text & val_text)
dup_val_test   = len(val_text & test_text)

print("train∩test:",dup_train_test)
print("train∩val:",dup_train_val)
print("val∩test:",dup_val_test)

train∩test: 39
train∩val: 38
val∩test: 44


In [16]:
vectorizer = TfidfVectorizer(max_features=5000)

train_vec = vectorizer.fit_transform(train_sent.text)
test_vec  = vectorizer.transform(test_sent.text)

sim = cosine_similarity(train_vec[:200],test_vec[:200])

near_duplicates = (sim>0.95).sum()

print("Near duplicates:",near_duplicates)

Near duplicates: 30


In [17]:
manifest = {
 "dataset":"UD Ukrainian ParlaMint",
 "strategy":"predefined_dataset_split",
 "unit_of_split":"sentence_id",

 "train_ids":"data/sample/splits_train_ids.txt",
 "validation_ids":"data/sample/splits_val_ids.txt",
 "test_ids":"data/sample/splits_test_ids.txt",

 "csv_files":{
     "train":"data/splits/train.csv",
     "validation":"data/splits/validation.csv",
     "test":"data/splits/test.csv"
 },

 "created":str(datetime.now())
}

with open("docs/splits_manifest_lab5.json","w") as f:
    json.dump(manifest,f,indent=2,ensure_ascii=False)

print("manifest створено")

manifest створено


In [18]:
report=f"""
# Leakage Risk Report — Lab5

Датасет: UD Ukrainian ParlaMint

Стратегія split:
використано офіційний train/dev/test split датасету.

Одиниця split:
sentence_id

Перевірки leakage:

Exact duplicates

train∩test: {dup_train_test}
train∩val: {dup_train_val}
val∩test: {dup_val_test}

Near duplicates:
{near_duplicates}

Висновок:
критичних ознак data leakage не виявлено.
"""

with open("docs/leakage_risk_report_lab5.md","w") as f:
    f.write(report)

In [19]:
summary=f"""
# Dataset Audit Summary — Lab5

Датасет: UD Ukrainian ParlaMint

Split strategy:
predefined dataset split

train / validation / test:
визначені у вихідному датасеті.

Leakage checks:
duplicate sentences
near-duplicate similarity

Результат:
ризик витоку даних низький.
"""

with open("docs/audit_summary_lab5.md","w") as f:
    f.write(summary)

In [20]:
split_code = """
import pandas as pd

def load_split(data_path):
    df = pd.read_csv(data_path)

    train = df[df["split"]=="train"]
    val   = df[df["split"]=="dev"]
    test  = df[df["split"]=="test"]

    return train, val, test


if __name__ == "__main__":

    train,val,test = load_split("data/processed_v2/processed_v2.csv")

    print("Train:",len(train))
    print("Validation:",len(val))
    print("Test:",len(test))
"""

with open("src/split.py","w") as f:
    f.write(split_code)

print("split.py створено")

split.py створено


In [22]:
from google.colab import files
import os

# Базова папка проекту
base_dir = "/content"

# Список всіх файлів ЛР5, які треба завантажити
files_to_download = [
    "src/split.py",
    "notebooks/lab5_split_leakage_checks.ipynb",
    "docs/splits_manifest_lab5.json",
    "docs/leakage_risk_report_lab5.md",
    "docs/audit_summary_lab5.md",
    "data/sample/splits_train_ids.txt",
    "data/sample/splits_val_ids.txt",
    "data/sample/splits_test_ids.txt",
    "data/splits/train.csv",
    "data/splits/validation.csv",
    "data/splits/test.csv"
]

# Завантажуємо файли по черзі
for f in files_to_download:
    file_path = os.path.join(base_dir, f)
    if os.path.exists(file_path):
        print(f"⬇️ Завантажено {file_path}")
        files.download(file_path)
    else:
        print(f"⚠️ Файл не знайдено: {file_path}")

⬇️ Завантажено /content/src/split.py


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⚠️ Файл не знайдено: /content/notebooks/lab5_split_leakage_checks.ipynb
⬇️ Завантажено /content/docs/splits_manifest_lab5.json


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️ Завантажено /content/data/splits/test.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>